# **업무 이메일 자동화 AI 파이프라인**
---

## Ⅰ. 환경 설정 및 데이터 로드

1. 패키지 설치 + Drive Mount

In [ ]:
!pip install sentence-transformers scikit-learn pandas numpy seaborn matplotlib

# 한글 폰트 설치 (Colab 세션 시작마다 실행 필요)
!apt-get update -qq
!apt-get install -y fonts-nanum -qq
!fc-cache -fv

from google.colab import drive
drive.mount('/content/drive')

2.  sys.path 등록 + 폴더 구조 확인

In [ ]:
import os, sys

SRC_DIR = "/content/drive/MyDrive/Capstone_AI2/src"

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from config import (
    BASE_DIR, DATA_DIR, MODEL_DIR, SRC_DIR,
    OUTPUT_DIR, FIGURES_DIR, REPORTS_DIR, LOG_DIR,
    DATASET_PATH, PAIRS_CSV_PATH,
    EMBEDDINGS_BASELINE_PATH, EMBEDDINGS_FINETUNED_PATH,
    SBERT_MODEL_PATH,
    DOMAIN_CLF_PATH, DOMAIN_LE_PATH,
    INTENT_CLF_PATH, INTENT_LE_PATH,
)

print("=" * 55)
print("폴더 구조 확인")
print("=" * 55)
folders = {
    "data/"           : DATA_DIR,
    "models/"         : MODEL_DIR,
    "src/"            : SRC_DIR,
    "outputs/figures/": FIGURES_DIR,
    "outputs/reports/": REPORTS_DIR,
    "outputs/logs/"   : LOG_DIR,
}
for name, path in folders.items():
    status = "✔" if os.path.exists(path) else "❌"
    print(f"  {status}  {name:<22} {path}")

print("\n" + "=" * 55)
print("src 파일 확인")
print("=" * 55)
for fname in ["config.py", "data_utils.py", "train_sbert.py",
              "train_domain.py", "train_intent.py",
              "evaluation.py", "inference.py"]:
    path   = os.path.join(SRC_DIR, fname)
    status = "✔" if os.path.exists(path) else "❌ 없음"
    print(f"  {status}  {fname}")

3. 데이터 로드 및 검증

In [ ]:
from data_utils import load_dataset

df = load_dataset()

print(f"\n[도메인별 샘플 수]")
print(df.groupby("domain")["intent"].count().to_string())
print(f"\n[도메인 × 인텐트 분포]")
print(df.groupby(["domain", "intent"]).size().reset_index(name="count").to_string(index=False))

## Ⅱ. SBERT 파인튜닝 및 임베딩 생성

**1. SBERT Fine-tuning**

In [ ]:
from train_sbert import run_sbert_finetuning

model = run_sbert_finetuning()

**2. 임베딩 생성 + data/ 저장**

In [ ]:
from train_sbert import generate_embeddings
from config import EMBEDDINGS_FINETUNED_PATH

X = generate_embeddings(
    texts    =df["email_text"].tolist(),
    save_path=EMBEDDINGS_FINETUNED_PATH,
)

print(f"\n임베딩 저장 경로 : {EMBEDDINGS_FINETUNED_PATH}")
print(f"임베딩 shape     : {X.shape}")

**3.  임베딩 품질 검증**

In [ ]:
from evaluation import validate_embeddings

validate_embeddings(X, df)

## Ⅲ. 분류기 학습 및 평가

**1. Domain Classifier 학습 + 평가**

In [ ]:
from train_domain import train_domain_classifier

domain_clf, le_domain = train_domain_classifier(X, df["domain"].values)

**2.  Intent Classifier 학습 + 평가**

In [ ]:
from train_intent import train_intent_classifiers

intent_classifiers, intent_encoders = train_intent_classifiers(X, df)

## Ⅳ. 추론 파이프라인 테스트

1. 추론 파이프라인 로드 + 테스트

In [ ]:
from inference import load_pipeline, predict_email

pipeline = load_pipeline()

test_emails = [
    "안녕하세요. 귀사의 서비스에 대한 견적서를 요청드립니다.",
    "채용 관련 문의드립니다. 개발자 포지션 지원이 가능한지 확인하고 싶습니다.",
    "시스템 오류가 발생했습니다. 로그인이 되지 않아 긴급 지원 요청드립니다.",
]

print("=" * 55)
print("추론 테스트")
print("=" * 55)
for email in test_emails:
    result = predict_email(email, pipeline)
    print(f"\n입력   : {email[:45]}...")
    print(f"  Domain : {result['domain']:<20} (confidence: {result['domain_confidence']})")
    print(f"  Intent : {result['intent']:<20} (confidence: {result['intent_confidence']})")
    print(f"  Low confidence 플래그 : {result['low_confidence']}")